# Milestone 3: Predictive Modeling

In this milestone, machine learning regression models are developed to estimate visa processing time based on historical application data.

The goal is to train predictive models that can learn relationships between applicant attributes, employer details, and historical trends to accurately estimate processing time.

The following models are implemented:

• Linear Regression  
• Random Forest Regressor  
• Gradient Boosting Regressor  

These models are evaluated using the following metrics:

• Mean Absolute Error (MAE)  
• Root Mean Squared Error (RMSE)  
• R² Score

Finally, the best performing model is selected and optimized using hyperparameter tuning.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("visa_dataset.csv")


## Feature Selection

In this step, relevant features are selected from the dataset that may influence visa processing time. These include employer information, wage details, and engineered features such as application month and country-specific average processing time.

The target variable is **processing_days**, which represents the number of days between visa application submission and final decision.

In [2]:
# convert date columns
df['case_received_date'] = pd.to_datetime(df['case_received_date'])
df['decision_date'] = pd.to_datetime(df['decision_date'])

# create month feature
df['month'] = df['case_received_date'].dt.month

country_avg = df.groupby('country_of_citizenship')['processing_days'].mean()

df['country_avg_processing'] = df['country_of_citizenship'].map(country_avg)


In [3]:
# Selecting features for model training

features = [
    'employer_num_employees',
    'employer_yr_estab',
    'pw_amount_9089',
    'wage_offer_from_9089',
    'month',
    'country_avg_processing'
]

X = df[features]
y = df['processing_days']

print("Feature matrix shape:", X.shape)
print("Target variable shape:", y.shape)

Feature matrix shape: (239091, 6)
Target variable shape: (239091,)


## Train-Test Split

The dataset is divided into two parts:

• Training set (80%) – used to train the machine learning models  
• Testing set (20%) – used to evaluate model performance on unseen data

This helps ensure that the model generalizes well and avoids overfitting.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

Training samples: (191272, 6)
Testing samples: (47819, 6)


## Linear Regression Model

Linear Regression is a simple baseline regression algorithm that models the relationship between input features and the target variable using a linear equation.

It assumes that the target variable can be predicted as a linear combination of the input features.

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Performance")
print("MAE:", mae_lr)
print("RMSE:", rmse_lr)
print("R2 Score:", r2_lr)

Linear Regression Performance
MAE: 111.57725991215163
RMSE: 196.48039651090718
R2 Score: 0.012109017907730668


## Random Forest Regressor

Random Forest is an ensemble learning algorithm that builds multiple decision trees and combines their predictions.

Advantages:

• Handles nonlinear relationships  
• Reduces overfitting compared to a single decision tree  
• Works well with complex datasets

In [6]:
import sklearn
from sklearn.ensemble import RandomForestRegressor

In [14]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=20,   # reduce trees
    max_depth=10,      # limit tree size
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Performance")
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R2 Score:", r2_rf)

Random Forest Performance
MAE: 89.52841960834881
RMSE: 168.47837036078533
R2 Score: 0.2736283669750845


## Gradient Boosting Regressor

Gradient Boosting is another ensemble learning method that builds models sequentially.

Each new model focuses on correcting the errors made by the previous models, leading to improved predictive performance.

In [8]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(random_state=42)

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)

mae_gb = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb = r2_score(y_test, y_pred_gb)

print("Gradient Boosting Performance")
print("MAE:", mae_gb)
print("RMSE:", rmse_gb)
print("R2 Score:", r2_gb)

Gradient Boosting Performance
MAE: 100.21688366365102
RMSE: 176.44457393290995
R2 Score: 0.2033140027578777


## Model Comparison

The performance of all models is compared using evaluation metrics to determine which algorithm provides the most accurate predictions.

In [15]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "Gradient Boosting"],
    "MAE": [mae_lr, mae_rf, mae_gb],
    "RMSE": [rmse_lr, rmse_rf, rmse_gb],
    "R2 Score": [r2_lr, r2_rf, r2_gb]
})

results

,Model,MAE,RMSE,R2 Score
0,Linear Regression,111.577260,196.480397,0.012109
1,Random Forest,89.528420,168.478370,0.273628
2,Gradient Boosting,100.216884,176.444574,0.203314


## Hyperparameter Tuning

Hyperparameter tuning is performed to improve the performance of the Random Forest model.

GridSearchCV is used to test multiple combinations of parameters and identify the best performing configuration.

In [16]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [100],
    'max_depth': [10]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=3
)

grid.fit(X_train, y_train)

print(grid.best_params_)

{'max_depth': 10, 'n_estimators': 100}


## Final Model Selection

Based on evaluation metrics and hyperparameter tuning results, the optimized Random Forest model is selected as the final predictive model for estimating visa processing time.

In [17]:
best_model = grid.best_estimator_

y_pred_final = best_model.predict(X_test)

print("Final Model Performance")

print("MAE:", mean_absolute_error(y_test, y_pred_final))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_final)))
print("R2 Score:", r2_score(y_test, y_pred_final))

Final Model Performance
MAE: 89.45168908486109
RMSE: 167.92014732444164
R2 Score: 0.27843379854639794


## Conclusion

In this milestone, multiple regression models were implemented to predict visa processing time.

The models evaluated include:

• Linear Regression  
• Random Forest Regressor  
• Gradient Boosting Regressor  

After comparing model performance using MAE, RMSE, and R² Score, the Random Forest model demonstrated superior predictive capability.

Hyperparameter tuning further improved the model performance, and the optimized Random Forest model was selected as the final model.

This trained model will be integrated into the processing time estimator system in the next milestone.

In [22]:
import pickle
pickle.dump(best_model, open("model.pkl", "wb"))